In [0]:
dbutils.widgets.text('date_param', '', label = None)

In [0]:
from datetime import datetime

if dbutils.widgets.get('date_param') == '':
    date_parameter = '20260101'
else:
    date_parameter = dbutils.widgets.get('date_param')

load_date = date_parameter
EDL_LoadDate = date_parameter

In [0]:
import os
from pyspark.sql.functions import lit, col
from datetime import datetime, timedelta
from pyspark.sql import functions as F

app_reg_app_id = os.environ['APP_REG_APP_ID']
ReadStorage = os.environ['GDP_STORAGE_NAME']
TenantId = os.environ['TENANT_ID']

service_credential = dbutils.secrets.get(scope="connectedsecrets", key = f"appreg-{app_reg_app_id}")

spark.conf.set("fs.azure.account.auth.type."+ReadStorage+".dfs.core.windows.net", "OAuth") 
spark.conf.set("fs.azure.account.oauth.provider.type."+ReadStorage+".dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider") 
spark.conf.set("fs.azure.account.oauth2.client.id."+ReadStorage+".dfs.core.windows.net", ""+app_reg_app_id+"") 
spark.conf.set("fs.azure.account.oauth2.client.secret."+ReadStorage+".dfs.core.windows.net", service_credential) 
spark.conf.set("fs.azure.account.oauth2.client.endpoint."+ReadStorage+".dfs.core.windows.net", "https://login.microsoftonline.com/"+TenantId+"/oauth2/token")

In [0]:
from datetime import datetime, timedelta
import re

planet_tables = [
    'Assessment'
    , 'ClientSnapShot'
    , 'ISA'
    , 'Questionnaire'
    , 'User'
]

for table in planet_tables:
    # get the most recent version available in gdp
    path = f'abfss://planet@edlcorestdeuprod0001.dfs.core.windows.net/{table}/'
    files = dbutils.fs.ls(path)
    version = max([int(re.search(r'/(\d+)/$', file.path).group(1)) for file in files if re.search(r'/(\d+)/$', file.path)])

    # load
    spark.read.parquet(f'abfss://planet@edlcorestdeuprod0001.dfs.core.windows.net/{table}/{version}/data/LOADED_DTS={load_date}*/*.parquet').createOrReplaceTempView(table)

planetclients

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW planetclients_historical AS
  SELECT DISTINCT
    CAST(GCID AS INT)
    , FullLegalName
    , Region
    , `Location`
    , BusinessLine
    , GlobalClientOwner
    , Sector
    , SubSector
    , CAST(NAICSCode AS INT)
    , NAICSDescription
    , IsListed AS Listed
    -- , UltimateParent
  FROM ClientSnapShot

In [0]:
from pyspark.sql.functions import lit, to_date

if EDL_LoadDate == '20260101':
    spark.sql('DROP TABLE IF EXISTS radar.planetclients_historical')
    spark.sql('SELECT * FROM planetclients_historical').withColumn('EDL_LoadDate', to_date(lit(EDL_LoadDate), 'yyyyMMdd')).write.mode('overwrite').saveAsTable('radar.planetclients_historical')

else:
    spark.sql('SELECT * FROM planetclients_historical').withColumn('EDL_LoadDate', to_date(lit(EDL_LoadDate), 'yyyyMMdd')).write.mode('append').saveAsTable('radar.planetclients_historical')

In [0]:
duplicated_uniquegcid = spark.sql(f"""
    SELECT GCID
    FROM radar.planetclients_historical
    WHERE DATE(EDL_LoadDate) = DATE('{EDL_LoadDate}')
    GROUP BY GCID
    HAVING count(*) > 1
""")

# check for duplication
if not duplicated_uniquegcid.isEmpty():
    raise Exception('Duplicated GCID in planetclients. Job stopped.')

planetcases

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW planetcases_historical AS
    SELECT DISTINCT
        t1.PrefixCaseID AS CaseID
        , CAST(t2.GCID AS INT)
        , COALESCE(t6.QCQuestionsTotal, 0) AS QCQuestionsTotal
        , t1.CompleteDate AS DateCompleted
        , t1.AssessmentCreatedDate AS DateCreated
        , t1.ExpirationDate
        , t3.FullName AS Assessor
        , t4.FullName AS Reviewer
        , t5.Applicable AS ISAApplicable
        , t5.Conclusion AS ISAConclusion
        , t1.PerformanceScore AS PerformanceScore
        , t1.PolicyScore
        , t1.Status AS StatusCode
                
        FROM Assessment t1
        LEFT JOIN ClientSnapShot t2 ON t1.ID = t2.Assessment_ClientSnapShot
        LEFT JOIN (SELECT * FROM User WHERE UserRole = 'Assessor') t3 ON t1.ID = t3.Account_Assessment
        LEFT JOIN (SELECT * FROM User WHERE UserRole = 'Reviewer') t4 ON t1.ID = t4.Account_Assessment
        LEFT JOIN ISA t5 ON t1.ID = t5.Assessment_ISA
        LEFT JOIN (
            SELECT 
                Questionnaire_Assessment
                , COUNT(*) AS QCQuestionsTotal 
            FROM Questionnaire
            WHERE AnswerOption IS NOT NULL
            GROUP BY Questionnaire_Assessment
        ) t6 ON t1.ID = t6.Questionnaire_Assessment

In [0]:
from pyspark.sql.functions import lit, to_date

if EDL_LoadDate == '20260101':
    spark.sql('DROP TABLE IF EXISTS radar.planetcases_historical')
    spark.sql('SELECT * FROM planetcases_historical').withColumn('EDL_LoadDate', to_date(lit(EDL_LoadDate), 'yyyyMMdd')).write.mode('overwrite').saveAsTable('radar.planetcases_historical')

else:
    spark.sql('SELECT * FROM planetcases_historical').withColumn('EDL_LoadDate', to_date(lit(EDL_LoadDate), 'yyyyMMdd')).write.mode('append').saveAsTable('radar.planetcases_historical')

In [0]:
duplicated_uniquecaseid = spark.sql(f"""
    SELECT CaseID
    FROM radar.planetcases_historical
    WHERE DATE(EDL_LoadDate) = DATE('{EDL_LoadDate}')
    GROUP BY CaseID 
    HAVING count(*) > 1
""")

# check for duplication
if not duplicated_uniquecaseid.isEmpty():
    raise Exception('Duplicated CaseID in planetcases. Job stopped.')

planetanswers

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW planetanswers_historical AS
    SELECT DISTINCT
        CONCAT(t1.PrefixCaseID, t2.QuestionID) AS ID
        , t1.PrefixCaseID AS CaseID
        , t2.QuestionID
        , t2.AnswerOption AS Answer
        , t2.PerformanceScore AS SubThemeScorePerformance
        , t2.PolicyScore AS SubThemeScorePolicy
             
    FROM Assessment t1
    LEFT JOIN Questionnaire t2 ON t1.ID = t2.Questionnaire_Assessment
    WHERE t2.AnswerOption IS NOT NULL

In [0]:
from pyspark.sql.functions import lit, to_date

if EDL_LoadDate == '2026-01-01':
    spark.sql('DROP TABLE IF EXISTS radar.planetanswers_historical')
    spark.sql('SELECT * FROM planetanswers_historical').withColumn('EDL_LoadDate', to_date(lit(EDL_LoadDate), 'yyyyMMdd')).write.mode('overwrite').saveAsTable('radar.planetanswers_historical')

else:
    spark.sql('SELECT * FROM planetanswers_historical').withColumn('EDL_LoadDate', to_date(lit(EDL_LoadDate), 'yyyyMMdd')).write.mode('append').saveAsTable('radar.planetanswers_historical') 

In [0]:
duplicated_uniqueid = spark.sql(f"""
    SELECT ID
    FROM radar.planetanswers_historical
    WHERE DATE(EDL_LoadDate) = DATE('{EDL_LoadDate}')
    GROUP BY ID 
    HAVING count(*) > 1
""")

# check for duplication
if not duplicated_uniqueid.isEmpty():
    raise Exception('Duplicated ID in planetanswers. Job stopped.')

planetquestions

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW planetquestions_historical AS
    SELECT DISTINCT
        t1.QuestionID
        , t1.QuestionTitle AS Question
        , t1.ThemeTitle AS Theme
        , t1.SubThemeOrPolicyTitle AS SubTheme
        , t1.QuestionType
    FROM Questionnaire t1
    JOIN (
        SELECT 
            QuestionID
            , MAX(QuestionnaireCreateData) AS MostRecentDate
        FROM 
            Questionnaire
        GROUP BY 
            QuestionID
    ) t2 ON t1.QuestionID = t2.QuestionID AND t1.QuestionnaireCreateData = t2.MostRecentDate

In [0]:
from pyspark.sql.functions import lit, to_date

if EDL_LoadDate == '2026-01-01':
    spark.sql('DROP TABLE IF EXISTS radar.planetquestions_historical')
    spark.sql('SELECT * FROM planetquestions_historical').withColumn('EDL_LoadDate', to_date(lit(EDL_LoadDate), 'yyyyMMdd')).write.mode('overwrite').saveAsTable('radar.planetquestions_historical')

else:
    spark.sql('SELECT * FROM planetquestions_historical').withColumn('EDL_LoadDate', to_date(lit(EDL_LoadDate), 'yyyyMMdd')).write.mode('append').saveAsTable('radar.planetquestions_historical')      

In [0]:
duplicated_uniquequestionid = spark.sql(f"""
    SELECT QuestionID
    FROM radar.planetquestions_historical
    WHERE DATE(EDL_LoadDate) = DATE('{EDL_LoadDate}')
    GROUP BY QuestionID 
    HAVING count(*) > 1
""")

# check for duplication
if not duplicated_uniquequestionid.isEmpty():
    raise Exception('Duplicated QuestionID in planetquestions. Job stopped.')